# XGBoost Probability Threshold Tuning — PharmShed
**Author:** Akhila Annireddy  
**Purpose:** Apply per-drug probability threshold tuning on top of already-trained XGBoost models.  
**Models used:** Baseline XGBoost (`xgboost_final_model.ubj`) and Sqrt-Weighted XGBoost (`xgboost_weighted_final_model.ubj`)  
**No retraining:** This notebook only changes how predictions are interpreted — the saved models are loaded and used as-is.  

## What Is Probability Threshold Tuning?

Normally XGBoost predicts whichever drug has the highest probability for each row.  
Example: if atorvastatin = 40% and ivermectin = 2%, it always predicts atorvastatin.  

With threshold tuning, we lower the bar for rare drugs.  
Example: if ivermectin's threshold is set to 1%, then its 2% probability is enough to predict it.  

**How thresholds are set:**  
We use inverse frequency — rarer drugs get lower thresholds so the model is more willing to predict them.  
Threshold for drug d = min_freq / freq(d), normalized so common drugs keep threshold = 1.0  

**What this trades off:**  
- Recall goes UP for rare drugs (model predicts them more often)  
- Precision goes DOWN (model may predict rare drugs when it shouldn't)  
- This is the right tradeoff for us — missing a drug (false negative) = underestimating pollution  

In [1]:
!pip install 'xgboost>=2.1.0' permetrics

In [2]:
# Load all required libraries.
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

print("XGBoost version:", xgb.__version__)

XGBoost version: 2.1.4


In [3]:
# Load the 2022 validation data — this is what we evaluate threshold tuning on.
# We use 2022 because it is the held-out test set per Ren's framework.
# The models were never trained on 2022 data so results are unbiased.

# Load integrated data just to fit LabelEncoder with same drug mapping as training
integrated_data = pd.read_csv('integrated_data.csv')
if 'Unnamed: 0' in integrated_data.columns:
    integrated_data = integrated_data.drop(columns=['Unnamed: 0'])

# Load 2022 test data
data_2022 = pd.read_csv('data_2022.csv')
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

print("Integrated data shape:", integrated_data.shape)
print("2022 data shape:", data_2022.shape)

Integrated data shape: (905728, 7)
2022 data shape: (175669, 7)


In [4]:
# Define columns and fit LabelEncoder on full integrated data.
# Must use same encoder as training to ensure consistent drug→integer mapping.

feature_cols     = ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity']
categorical_cols = ['Sex', 'Insurance_coverage', 'Race_ethnicity']
target_col       = 'Drug'

le = LabelEncoder()
le.fit(integrated_data[target_col])

print("Unique classes:", len(le.classes_))

# Filter 2022 data to only drugs the model knows
known_drugs          = set(le.classes_)
unseen_drugs         = set(data_2022[target_col].unique()) - known_drugs
data_2022_filtered   = data_2022[data_2022[target_col].isin(known_drugs)].copy()
print(f"Unseen drugs dropped: {len(unseen_drugs)}")
print(f"2022 rows after filtering: {len(data_2022_filtered):,}")

# Prepare 2022 features and labels
X_2022 = data_2022_filtered[feature_cols].copy()
for col in categorical_cols:
    X_2022[col] = X_2022[col].astype('category')

y_2022_encoded = le.transform(data_2022_filtered[target_col])
d2022 = xgb.DMatrix(X_2022, label=y_2022_encoded, enable_categorical=True)

Unique classes: 217
Unseen drugs dropped: 0
2022 rows after filtering: 175,669


In [5]:
# Compute per-drug probability thresholds based on drug frequency.
#
# LOGIC:
# Common drugs (high frequency) get threshold = 1.0 — predict only when clearly dominant
# Rare drugs (low frequency) get threshold < 1.0 — predict even at lower probabilities
#
# Formula: threshold(d) = sqrt(freq(d) / max_freq)
# - Most common drug gets threshold = sqrt(1.0) = 1.0
# - Rarest drug gets threshold = sqrt(29/97497) = sqrt(0.0003) = 0.017
#
# We use sqrt to soften the threshold ratio — same reasoning as sqrt weights.
# Full inverse would make rare drug thresholds too low, flooding predictions.
#
# To apply: divide each drug's probability by its threshold, then argmax.
# This effectively boosts rare drug probabilities relative to common ones.

drug_counts = integrated_data['Drug'].value_counts()
max_freq    = drug_counts.max()

# Build threshold array indexed by encoded drug integer
thresholds = np.array([
    np.sqrt(drug_counts.get(drug, 1) / max_freq)
    for drug in le.classes_
])

print("Threshold stats:")
print(f"  Max threshold (most common drug):  {thresholds.max():.4f}")
print(f"  Min threshold (rarest drug):       {thresholds.min():.4f}")
print(f"  Ratio max/min:                     {thresholds.max()/thresholds.min():.1f}x")

print("\nExample thresholds:")
for drug in ['no prescriptions', 'atorvastatin', 'lisinopril', 'metformin',
             'ivermectin', 'piroxicam', 'gentamicin']:
    if drug in le.classes_:
        idx = np.where(le.classes_ == drug)[0][0]
        print(f"  {drug:25s}: {thresholds[idx]:.4f}")

Threshold stats:
  Max threshold (most common drug):  1.0000
  Min threshold (rarest drug):       0.0172
  Ratio max/min:                     58.0x

Example thresholds:
  no prescriptions         : 1.0000
  atorvastatin             : 0.6289
  lisinopril               : 0.6064
  metformin                : 0.5886
  ivermectin               : 0.0172
  piroxicam                : 0.0250
  gentamicin               : 0.0256


In [6]:
# Helper function to apply threshold tuning and compute all metrics.
# Reused for both baseline and weighted models.

def apply_threshold_and_evaluate(model, dmatrix, y_true, thresholds, le, label):
    """
    Apply probability threshold tuning to a trained XGBoost model.
    
    Steps:
    1. Get softprob output — probabilities for all 217 drugs per row
    2. Divide each drug's probability by its threshold
       (this boosts rare drug probabilities relative to common ones)
    3. Take argmax of adjusted probabilities as final prediction
    4. Compute all required metrics and compare to hard predictions
    """
    n_classes = len(le.classes_)
    
    # Get probability outputs — softprob returns shape (n_rows, n_classes)
    raw_probs = model.predict(dmatrix)
    
    # softprob returns flat array of size n_rows * n_classes — reshape to 2D
    n_rows = len(y_true)
    if raw_probs.ndim == 1:
        probs = raw_probs.reshape(n_rows, n_classes)
    else:
        probs = raw_probs
    
    # Verify shape is correct
    assert probs.shape == (n_rows, n_classes), f"Expected ({n_rows}, {n_classes}), got {probs.shape}"
    
    print(f"\n{'='*50}")
    print(f"{label}")
    print(f"{'='*50}")
    print(f"Probability matrix shape: {probs.shape}")
    
    # --- Hard predictions (no threshold) ---
    y_pred_hard = np.argmax(probs, axis=1)
    
    acc_hard   = accuracy_score(y_true, y_pred_hard)
    kappa_hard = cohen_kappa_score(y_true, y_pred_hard)
    mcc_hard   = matthews_corrcoef(y_true, y_pred_hard)
    ev_hard    = ClassificationMetric(y_true, y_pred_hard)
    macro_recall_hard = ev_hard.recall_score(average='macro')
    macro_f2_hard     = ev_hard.fbeta_score(beta=2, average='macro')
    micro_recall_hard = ev_hard.recall_score(average='micro')
    
    print("\nHard predictions (no threshold):")
    print(f"  Accuracy:     {acc_hard:.4f}")
    print(f"  MCC:          {mcc_hard:.4f}")
    print(f"  Macro Recall: {macro_recall_hard:.4f}")
    print(f"  Micro Recall: {micro_recall_hard:.4f}")
    print(f"  Macro F2:     {macro_f2_hard:.4f}")
    
    # --- Threshold-adjusted predictions ---
    # Divide each column (drug) by its threshold
    # Rare drugs have low thresholds so their adjusted score is higher
    # = model becomes more willing to predict rare drugs
    adjusted_probs = probs / thresholds[np.newaxis, :]  # broadcast across rows
    y_pred_thresh  = np.argmax(adjusted_probs, axis=1)
    
    acc_thresh   = accuracy_score(y_true, y_pred_thresh)
    kappa_thresh = cohen_kappa_score(y_true, y_pred_thresh)
    mcc_thresh   = matthews_corrcoef(y_true, y_pred_thresh)
    ev_thresh    = ClassificationMetric(y_true, y_pred_thresh)
    macro_recall_thresh = ev_thresh.recall_score(average='macro')
    macro_f2_thresh     = ev_thresh.fbeta_score(beta=2, average='macro')
    micro_recall_thresh = ev_thresh.recall_score(average='micro')
    macro_prec_thresh   = ev_thresh.precision_score(average='macro')
    
    print("\nThreshold-adjusted predictions:")
    print(f"  Accuracy:        {acc_thresh:.4f}  (change: {acc_thresh-acc_hard:+.4f})")
    print(f"  MCC:             {mcc_thresh:.4f}  (change: {mcc_thresh-mcc_hard:+.4f})")
    print(f"  Macro Recall:    {macro_recall_thresh:.4f}  (change: {macro_recall_thresh-macro_recall_hard:+.4f})")
    print(f"  Micro Recall:    {micro_recall_thresh:.4f}  (change: {micro_recall_thresh-micro_recall_hard:+.4f})")
    print(f"  Macro Precision: {macro_prec_thresh:.4f}")
    print(f"  Macro F2:        {macro_f2_thresh:.4f}  (change: {macro_f2_thresh-macro_f2_hard:+.4f})")
    
    # Per-drug recall after thresholding
    report_thresh = classification_report(
        y_true, y_pred_thresh,
        labels=np.arange(n_classes),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_metrics = pd.DataFrame([
        {'Drug': drug,
         'Recall_thresh':    report_thresh[drug]['recall'],
         'Precision_thresh': report_thresh[drug]['precision'],
         'F2_thresh':        report_thresh[drug]['f1-score'],
         'Support':          report_thresh[drug]['support']}
        for drug in le.classes_ if drug in report_thresh
    ]).sort_values('Recall_thresh', ascending=False)
    
    print("\nTop 15 drugs by recall after threshold tuning:")
    print(drug_metrics.head(15).to_string(index=False))
    
    print("\nBottom 10 drugs — checking if previously-zero drugs now get predicted:")
    print(drug_metrics.tail(10).to_string(index=False))
    
    # How many drugs went from 0 recall to >0 recall
    newly_predicted = drug_metrics[
        (drug_metrics['Recall_thresh'] > 0)
    ]
    print(f"\nDrugs with recall > 0 after thresholding: {len(newly_predicted)} / {n_classes}")
    
    return {
        'label':                label,
        'accuracy_hard':        acc_hard,
        'macro_recall_hard':    macro_recall_hard,
        'macro_f2_hard':        macro_f2_hard,
        'accuracy_thresh':      acc_thresh,
        'mcc_thresh':           mcc_thresh,
        'macro_recall_thresh':  macro_recall_thresh,
        'micro_recall_thresh':  micro_recall_thresh,
        'macro_f2_thresh':      macro_f2_thresh,
        'drugs_with_recall':    len(newly_predicted),
    }, drug_metrics

## Part 1 — Baseline XGBoost + Threshold Tuning

Load the baseline XGBoost final model (trained without class weights) and apply threshold tuning.
This model was strong on common drugs (atorvastatin 44% recall) but had 0% recall on 150+ drugs.

In [7]:
# Load baseline XGBoost final model.
# Model was saved with multi:softmax (outputs winning class integer).
# We override to multi:softprob so predict() returns a full probability
# distribution across all 217 classes — shape (n_rows, 217).
# Tree structure is unchanged — only the output format changes.

baseline_model = xgb.Booster()
baseline_model.load_model('xgboost_final_model.ubj')
baseline_model.set_param({'objective': 'multi:softprob', 'num_class': len(le.classes_)})
print("Baseline model loaded with softprob output.")

# Apply threshold tuning and evaluate
baseline_results, baseline_drug_metrics = apply_threshold_and_evaluate(
    model      = baseline_model,
    dmatrix    = d2022,
    y_true     = y_2022_encoded,
    thresholds = thresholds,
    le         = le,
    label      = 'BASELINE XGBoost — Threshold Tuning on MEPS 2022'
)

# Save per-drug metrics
baseline_drug_metrics.to_csv('xgboost_baseline_threshold_per_drug.csv', index=False)
print("\nBaseline threshold per-drug metrics saved.")

Baseline model loaded with softprob output.

BASELINE XGBoost — Threshold Tuning on MEPS 2022
Probability matrix shape: (175669, 217)

Hard predictions (no threshold):
  Accuracy:     0.0954
  MCC:          0.0562
  Macro Recall: 0.0101
  Micro Recall: 0.0954
  Macro F2:     0.0072

Threshold-adjusted predictions:
  Accuracy:        0.0664  (change: -0.0290)
  MCC:             0.0452  (change: -0.0110)
  Macro Recall:    0.0188  (change: +0.0087)
  Micro Recall:    0.0664  (change: -0.0290)
  Macro Precision: 0.0144
  Macro F2:        0.0149  (change: +0.0077)

Top 15 drugs by recall after threshold tuning:
              Drug  Recall_thresh  Precision_thresh  F2_thresh  Support
  no prescriptions       0.577988          0.279824   0.377088  10040.0
        tamsulosin       0.246796          0.044072   0.074788   2107.0
   methylphenidate       0.231330          0.149764   0.181818   1098.0
 ethinyl estradiol       0.227273          0.013774   0.025974     22.0
dexmethylphenidate       

## Part 2 — Sqrt-Weighted XGBoost + Threshold Tuning

Load the sqrt-weighted XGBoost final model and apply threshold tuning on top.
This model already had better macro recall (1.6% vs 0.9%) from class weighting.
Threshold tuning on top should push it further.

In [8]:
# Load sqrt-weighted XGBoost final model.
# Same softprob override as baseline — need full probability distribution.

weighted_model = xgb.Booster()
weighted_model.load_model('xgboost_weighted_final_model.ubj')
weighted_model.set_param({'objective': 'multi:softprob', 'num_class': len(le.classes_)})
print("Weighted model loaded with softprob output.")

# Apply threshold tuning and evaluate
weighted_results, weighted_drug_metrics = apply_threshold_and_evaluate(
    model      = weighted_model,
    dmatrix    = d2022,
    y_true     = y_2022_encoded,
    thresholds = thresholds,
    le         = le,
    label      = 'WEIGHTED XGBoost — Threshold Tuning on MEPS 2022'
)

# Save per-drug metrics
weighted_drug_metrics.to_csv('xgboost_weighted_threshold_per_drug.csv', index=False)
print("\nWeighted threshold per-drug metrics saved.")

Weighted model loaded with softprob output.

WEIGHTED XGBoost — Threshold Tuning on MEPS 2022
Probability matrix shape: (175669, 217)

Hard predictions (no threshold):
  Accuracy:     0.0932
  MCC:          0.0540
  Macro Recall: 0.0099
  Micro Recall: 0.0932
  Macro F2:     0.0071

Threshold-adjusted predictions:
  Accuracy:        0.0638  (change: -0.0294)
  MCC:             0.0435  (change: -0.0105)
  Macro Recall:    0.0182  (change: +0.0083)
  Micro Recall:    0.0638  (change: -0.0294)
  Macro Precision: 0.0145
  Macro F2:        0.0147  (change: +0.0076)

Top 15 drugs by recall after threshold tuning:
              Drug  Recall_thresh  Precision_thresh  F2_thresh  Support
  no prescriptions       0.569323          0.279183   0.374648  10040.0
   methylphenidate       0.274135          0.179916   0.217250   1098.0
 ethinyl estradiol       0.227273          0.011547   0.021978     22.0
        tamsulosin       0.202658          0.043363   0.071441   2107.0
dexmethylphenidate       

In [9]:
# Final comparison table across all versions.
# This is the summary table that goes into the paper.

comparison = pd.DataFrame([
    {
        'Model':          'XGBoost Baseline',
        'Accuracy':        baseline_results['accuracy_hard'],
        'Macro_Recall':    baseline_results['macro_recall_hard'],
        'Macro_F2':        baseline_results['macro_f2_hard'],
        'Drugs_Recalled':  'N/A'
    },
    {
        'Model':          'XGBoost Baseline + Threshold',
        'Accuracy':        baseline_results['accuracy_thresh'],
        'Macro_Recall':    baseline_results['macro_recall_thresh'],
        'Macro_F2':        baseline_results['macro_f2_thresh'],
        'Drugs_Recalled':  baseline_results['drugs_with_recall']
    },
    {
        'Model':          'XGBoost Sqrt Weighted',
        'Accuracy':        weighted_results['accuracy_hard'],
        'Macro_Recall':    weighted_results['macro_recall_hard'],
        'Macro_F2':        weighted_results['macro_f2_hard'],
        'Drugs_Recalled':  'N/A'
    },
    {
        'Model':          'XGBoost Sqrt Weighted + Threshold',
        'Accuracy':        weighted_results['accuracy_thresh'],
        'Macro_Recall':    weighted_results['macro_recall_thresh'],
        'Macro_F2':        weighted_results['macro_f2_thresh'],
        'Drugs_Recalled':  weighted_results['drugs_with_recall']
    },
])

print("="*70)
print("FULL COMPARISON — All XGBoost Versions on MEPS 2022")
print("="*70)
print(comparison.to_string(index=False))

comparison.to_csv('xgboost_all_versions_comparison.csv', index=False)
print("\nComparison table saved to xgboost_all_versions_comparison.csv")

FULL COMPARISON — All XGBoost Versions on MEPS 2022
                            Model  Accuracy  Macro_Recall  Macro_F2 Drugs_Recalled
                 XGBoost Baseline  0.095429      0.010107  0.007196            N/A
     XGBoost Baseline + Threshold  0.066415      0.018831  0.014867             97
            XGBoost Sqrt Weighted  0.093198      0.009896  0.007061            N/A
XGBoost Sqrt Weighted + Threshold  0.063751      0.018200  0.014710            102

Comparison table saved to xgboost_all_versions_comparison.csv


## Summary of Outputs

| File | Contents |
|------|----------|
| `xgboost_baseline_threshold_per_drug.csv` | Per-drug recall after threshold tuning on baseline model |
| `xgboost_weighted_threshold_per_drug.csv` | Per-drug recall after threshold tuning on weighted model |
| `xgboost_all_versions_comparison.csv` | Full comparison table across all 4 XGBoost versions |

## Key Metric to Watch
**`Drugs_Recalled`** — how many of the 217 drugs have recall > 0 after threshold tuning.  
Baseline XGBoost had 0% recall on 150+ drugs.  
If threshold tuning pushes even 50 more drugs above 0% recall, that is a meaningful improvement for the ensemble.